# Aufgabe 4 · Jahreszählungen mit TinyFlux

[Aufgabenstellung](README.md) · [Walkthrough](WALKTHROUGH.md) · [Microblogging](../../demos/microblogging/04_tinyflux.ipynb) · [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md)

Die Forschungsredaktion möchte eine Grafik zum zeitlichen Verlauf ihres Katalogbestands veröffentlichen. Dein Auftrag: Speichere Jahreszählungen, frage ein Zeitfenster ab und formuliere eine Aussage, die der Bestand tatsächlich trägt.

Ergänze die mit TODO markierten Stellen in A und B. Unbearbeitete Zellen melden OFFEN und können von oben nach unten ausgeführt werden. C ist eine optionale Vertiefung. Deine Interpretation hältst Du in den markierten Markdown-Zellen fest.

## Ausgangslage und Zähleinheit

Die vorbereitete CSV enthält **820 Punkte: 82 Jahre × 10 Katalogstellen**. Ihre Zählfelder ergeben zusammen **292 geeignete Katalogeinträge**. 83 der insgesamt 375 Einträge besitzen keine eindeutige Jahreszuordnung und bleiben von dieser Reihe ausgeschlossen. Für diese Aufgabe sind keine früheren Workshopresultate und kein neuer Download nötig.

`period_start` markiert mit dem 1. Januar den Beginn des Jahresintervalls. Es ist kein genaues Ereignisdatum. `date_basis=catalog_incident_year` bezeichnet die verwendete Jahresangabe des Katalogs. Die Zählgrösse ist ein Katalogeintrag, nicht ein einzigartiges Ereignis oder eine bestätigte Sichtung. Das letzte Jahr 2026 ist vom Abrufstand abhängig und kein vollständig beobachtetes Kalenderjahr.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display
from tinyflux import Point, TinyFlux, TimeQuery, TagQuery, FieldQuery

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/tinyflux_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from tinyflux_workshop import (csv_rows, parse_time, database_path, prepare_points,
    replace_snapshot, search_points, points_frame, annual_inputs, agency_names,
    yearly_frame, daily_posts)
Time, Tag, Field = TimeQuery(), TagQuery(), FieldQuery()
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.figsize": (10, 4.3), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

SOLUTION = False
rows = annual_inputs()
names = agency_names()
fbi_id = next(key for key, name in names.items() if name == "FBI")
print("Eingangspunkte:", len(rows), "– geeignete Einträge:", sum(int(r["catalog_entries"]) for r in rows))
print("Arbeitsdatei:", database_path("uap", solution=SOLUTION))
display(pd.DataFrame(rows).head())

## A · Aus einer Jahreszählung einen Punkt bauen

Ergänze `time`, `tags` und `fields` in `make_annual_point`. Verwende `parse_time` für den vorhandenen Zeitstempel und eine numerische Ganzzahl für `catalog_entries`. Die Tags sollen `agency_id`, `date_basis` und `snapshot_sha256` erhalten. Der Snapshot-Hash verankert die abgeleitete Sicht im festgehaltenen Katalogbestand.

Überlege vor dem Import: Warum dürfen die zehn Punkte eines Jahres denselben Zeitstempel haben? Warum ist die Stelle ein Tag und die Anzahl ein Field?

In [ ]:
def make_annual_point(row):
    # TODO A: Zeitpunkt, drei Tags und numerisches Zählfeld ergänzen.
    time = None
    tags = None
    fields = None
    if time is None or tags is None or fields is None:
        return None
    return Point(time=time, measurement="uap_annual_catalog", tags=tags, fields=fields)

annual_points = [make_annual_point(row) for row in rows]
model_ready = all(isinstance(p, Point) for p in annual_points)
if model_ready:
    # Gegen die jeweilige Eingabe prüfen, nicht nur gegen eine Gesamtsumme.
    for row, point in zip(rows, annual_points):
        assert point.time == parse_time(row["period_start"]), "Zeitmarke prüfen."
        assert point.tags == {key: row[key] for key in ["agency_id", "date_basis", "snapshot_sha256"]}, "Tags prüfen."
        assert point.fields == {"catalog_entries": int(row["catalog_entries"])}, "Zählfeld prüfen."
    imported = replace_snapshot("uap", solution=SOLUTION, points=annual_points)
    repeated = replace_snapshot("uap", solution=SOLUTION, points=annual_points)
    assert imported == repeated and imported["points"] == 820
    stored_points = search_points("uap", solution=SOLUTION)
    assert sum(int(p.fields["catalog_entries"]) for p in stored_points) == 292
    print("A OK – 820 Punkte, Summe 292; Wiederholung ohne Doppelzählung.")
else:
    print("OFFEN: Ergänze make_annual_point in A.")

Der Import ersetzt nur die ausgewählte TinyFlux-Arbeitsdatei. Er prüft eine neue Datei nach erneutem Öffnen, bevor er die bisherige Version ersetzt. Eigene Änderungen innerhalb der gewählten Datei werden zurückgesetzt. Aufgabe und Musterlösung verwenden verschiedene Dateien; beide greifen auf dieselben unveränderten Eingaben zurück.

**Deine Modellbegründung:**

*TODO: Begründe Tags, Field und gemeinsame Zeitstempel in eigenen Worten.*

## B · Ein Zeitfenster für die Redaktion

Wie viele geeignete **FBI-Katalogeinträge** entfallen im Snapshot auf die Jahre **2020 bis einschliesslich 2023**?

1. Ergänze einen TinyFlux-Filter mit Stelle, eingeschlossener Untergrenze und ausgeschlossener Obergrenze: `[2020-01-01, 2024-01-01)` in UTC.
2. Summiere die `catalog_entries`-Felder der gefundenen Punkte. Die Anzahl der Punkte ist noch nicht die gesuchte Anzahl Katalogeinträge.
3. Ergänze in B3 die Beschriftung der y-Achse mit der korrekten Zähleinheit. Lies die Jahrestabelle und die Grafik; formuliere die Antwort mit Stelle, Intervall und Datumsgrundlage.

Die Grafik zeigt zusätzlich die angrenzenden Jahre 2018–2026. Orange markiert Dein Abfragefenster; Werte ausserhalb dieses Fensters dienen der Einordnung. Die vollständige FBI-Reihe bleibt als Tabelle verfügbar.

In [ ]:
since = datetime(2020, 1, 1, tzinfo=timezone.utc)
until = datetime(2024, 1, 1, tzinfo=timezone.utc)
# TODO B1: Bedingungen mit & und Klammern verknüpfen.
annual_query = None
window_points = []
window_total = None
if model_ready and annual_query is not None:
    window_points = search_points("uap", annual_query, solution=SOLUTION)
    # TODO B2: Zählwerte der gefundenen Punkte summieren.
    window_total = None
    display(yearly_frame(window_points).assign(agency="FBI")[["year", "agency", "catalog_entries"]])
    if window_total is None:
        print("OFFEN: Ergänze die Summe in B2.")
    else:
        assert len(window_points) == 4 and window_total == 25, "Stelle, Zeitgrenzen und Zählfeld prüfen."
        assert all(p.tags["agency_id"] == fbi_id and since <= p.time < until for p in window_points)
        print("B OK –", len(window_points), "Jahrespunkte ergeben", window_total, "geeignete Katalogeinträge.")
else:
    print("OFFEN: Zuerst A und den Filter in B1 ergänzen.")

In [ ]:
# TODO B3: y-Achse mit der fachlich richtigen Zähleinheit beschriften.
chart_y_label = None
if model_ready and chart_y_label is not None:
    fbi_series = yearly_frame(search_points("uap", Tag.agency_id == fbi_id, solution=SOLUTION))
    plot_series = fbi_series[fbi_series["year"] >= 2018].copy()
    in_window = (plot_series["year"] >= since.year) & (plot_series["year"] < until.year)
    fig, ax = plt.subplots(layout="constrained")
    ax.bar(plot_series["year"], plot_series["catalog_entries"],
           color=["#cb712f" if selected else "#94adb8" for selected in in_window])
    ax.set(title="FBI-Katalogeinträge nach Katalog-Ereignisjahr", xlabel="Jahr laut Katalog",
           ylabel=chart_y_label, xticks=plot_series["year"])
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.text(0, -0.24, "Orange: 2020–2023 · Jahreszählung im fixierten Snapshot · 2026 unvollständig",
            transform=ax.transAxes, fontsize=10)
    plt.show()
    display(fbi_series.tail(9))
else:
    print("OFFEN: Für die Grafik A und die Achsenbeschriftung in B3 ergänzen.")

### Interpretation mit Aktenbeleg

Öffne [FBI-UAP-D024](../../data/raw/originals/FBI-UAP-D024.pdf), Seiten 1–3, und [FBI-UAP-D026](../../data/raw/originals/FBI-UAP-D026.pdf), Seiten 1–3. Der [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md) hilft bei der Einordnung.

- D024 trägt im Katalog `2002, 2023-2024`. Warum verteilen wir diesen Eintrag nicht automatisch auf mehrere Jahre?
- D026 wird im Katalog 2023 zugeordnet; das Interview stammt aus Januar 2026. Was würde eine Gruppierung nach Interviewjahr verändern?
- Was bedeutet ein Nullwert im Diagramm? Was darf die Redaktion aus einem hohen Balken nicht folgern?

**Deine Veröffentlichungsaussage und Begründung:**

*TODO: Formuliere die Aussage; ergänze Deine Erklärung mit PDF-Seitenbeleg und trenne Datumsgrundlagen, Nullwerte und nicht zugeordnete Einträge.*

## C · Nullpunkte und gröbere Intervalle — optionale Vertiefung

Die folgende TinyFlux-Abfrage filtert zusätzlich auf ein numerisches Feld. Damit verschwinden Nullpunkte aus der Ergebnismenge, ohne dass sich deren Zählsumme ändert. Der Filter erzeugt keine vollständigere Datenbasis.

Die anschliessende Dekadenaggregation ist pandas-Verarbeitung. Die erste Dekade ist nur ab 1945, die letzte nur bis zum Snapshot in 2026 abgedeckt. Dekadensummen sind deshalb ohne weitere Beurteilung keine vergleichbaren Raten.

In [ ]:
if model_ready:
    positive_points = search_points("uap", Field.catalog_entries > 0, solution=SOLUTION)
    all_years = yearly_frame(search_points("uap", solution=SOLUTION))
    decade_totals = all_years.assign(decade=(all_years["year"] // 10) * 10).groupby("decade")["catalog_entries"].sum()
    assert int(decade_totals.sum()) == 292
    assert sum(int(p.fields["catalog_entries"]) for p in positive_points) == 292
    print("Alle Punkte:", len(all_years), "– positive Punkte:", len(positive_points))
    print("Explizite Nullpunkte:", int((all_years["catalog_entries"] == 0).sum()))
    display(decade_totals.rename("Geeignete Katalogeinträge").to_frame())
else:
    print("OFFEN: C kann nach dem Import aus A ausgeführt werden.")

## Abschluss und Modellvergleich

Halte in der gemeinsamen Vergleichstabelle fest: Welche Frage unterstützt die Jahresreihe direkt? Welche Details gingen bei der Aggregation verloren? Wo bleibt der führende Katalog, wenn die Redaktion einzelne Akten prüfen möchte?

TinyFlux bietet hier Zeit-, Tag- und Feldfilter über eine lokale Datei. Gruppierungen und Grafik entstehen in Python/pandas. SQL kann ebenfalls Zeitreihen speichern und Zeitfenster abfragen. Aus diesem kleinen Beispiel folgt keine allgemeine Leistungsrangliste. Für Produktion müssten unter anderem Parallelzugriffe, Datenmenge, Aufbewahrung und benötigte Abfragen gesondert beurteilt werden.

*TODO: Ergänze Deinen begründeten Modellvergleich.*

[Modell und Import](../../schemas/tinyflux/README.md)